In [28]:
# from google.colab import files
# uploaded = files.upload()


# !unzip ml-prague-2026_colab.zip
# !cp -a ml-prague-2026_colab/* ./

# %pip uninstall torchvision torchaudio -y
# %pip install -r requirements.txt

# Anomaly Detection on Graphs — Hands-on Workshop

In this notebook you will apply several anomaly detection methods to the **YelpChi** fraud-review dataset.  
The dataset contains Yelp reviews with hand-crafted features and a binary `spam` label. Reviews are connected through edges, forming a graph.

**Models we will train & compare:**
1. **Isolation Forest** — a classical tree-based anomaly detector (no graph structure)
2. **Two step approach** — learn node embeddings via link prediction GCN, then flag anomalies
3. **DOMINANT** — autoencoder-based graph anomaly detector (PyGOD)
4. **GAD-NR** — neighbourhood-reconstruction graph anomaly detector (PyGOD)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import classification_report

from ml_prague_2026.evaluation import compare_models, evaluate_model

from pygod.detector import DOMINANT, GADNR

import torch
import torch_geometric.nn as nn
import torch_geometric.utils as utils
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.seed import seed_everything


seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [41]:
YELP_CHI_PATH = 'data/yelpchi/yelpchi.parquet'

YELP_CHI_EDGE_RSR_PATH = 'data/yelpchi/yelpchi_rsr.npy'
YELP_CHI_EDGE_RTR_PATH = 'data/yelpchi/yelpchi_rtr.npy'
YELP_CHI_EDGE_RUR_PATH = 'data/yelpchi/yelpchi_rur.npy'

## 1. Data Loading

We start by loading the preprocessed YelpChi dataset (reviews with features) and the three edge lists that connect reviews sharing the same restaurant, timestamp, or user.

---

In [42]:
yelp_chi = pd.read_parquet(YELP_CHI_PATH)
yelp_chi.head(2)

,date,review_id,user_id,product_id,spam,useful,funny,cool,stars,review,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,f_9,f_10,f_11,f_12,f_13,f_14,f_15,f_16,f_17,f_20,f_21,f_22,f_23,f_24,f_25,f_26,f_27,f_28,f_29,f_30,f_31,review_idx
0,2011-06-08,MyNjnxzZVTPq,IFTr6_6NI4CgCVavIL9k5g,tQfLGoolUMu2J0igcWcoZg,0,28,11,18,5,Let me begin by saying that there are two kind...,0.022376,0.070495,0.428682,0.999985,0.999985,0.398457,0.823592,0.497025,0.965457,0.150263,0.999985,0.583218,0.583916,0.381482,0.381646,0.999974,0.643092,0.999974,0.755169,0.770512,0.94806,0.867772,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512,0
1,2011-08-30,BdD7fsPqHQL73hwENEDT-Q,c_-hF15XgNhlyy_TqzmdaA,tQfLGoolUMu2J0igcWcoZg,0,4,3,0,3,The only place inside the Loop that you can st...,0.024928,0.999985,0.999985,0.999985,0.999985,0.398457,0.999985,0.941257,0.217850,0.098019,0.999985,0.999985,1.000000,0.381482,0.381646,0.999974,0.776791,0.156924,0.999974,0.932113,0.94806,0.826367,0.995025,0.910448,0.079602,0.00995,0.014925,0.59204,0.139303,0.497512,1


In [43]:
edges = np.concatenate((
    np.load(YELP_CHI_EDGE_RSR_PATH, allow_pickle=True),
    np.load(YELP_CHI_EDGE_RTR_PATH, allow_pickle=True),
    np.load(YELP_CHI_EDGE_RUR_PATH, allow_pickle=True)
), axis=0)

## 2. Isolation Forest

[Isolation Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html) is a tree-based anomaly detector that works on tabular features **without** graph structure.  
It isolates anomalies by randomly partitioning features — anomalies require fewer splits, resulting in shorter average path lengths.

---

In [44]:
# Prepare features and labels
feature_cols = [c for c in yelp_chi.columns if c.startswith('f_')]
X = yelp_chi[feature_cols].values
y_true = yelp_chi['spam'].values

# Load train/test split from file (to ensure consistency with supervised notebook)
data_split = np.load("data/yelpchi/yelpchi_split.npz")
train_idx = np.concatenate([data_split['train'], data_split['val']])
test_idx = data_split['test']

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_true[train_idx], y_true[test_idx]

We use the known **contamination rate** (fraction of spam) so the model threshold matches the true anomaly ratio.

In [45]:
# Contamination approximates the fraction of anomalies (spam) in the dataset
contamination = y_train.mean()
print(f'Contamination (proportion of anomalies): {contamination:.4f}')

Contamination (proportion of anomalies): 0.1453


In [52]:
# TASK 1: experiment with the contamination parameter.
# - Try different values for contamination (e.g., 0.1, 0.2...)
# - Observe how it affects the recall and precision of the model on anomaly class

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)

iso_forest.fit(X_train)

IsolationForest(contamination=np.float64(0.14531309504379522), n_estimators=200,
                n_jobs=-1, random_state=42)

In [53]:
# IsolationForest returns -1 for anomalies, 1 for normal
y_pred_test = (iso_forest.predict(X_test) == -1).astype(int)

# Compute anomaly scores — negate score_samples so that higher = more anomalous
y_scores_test = -iso_forest.score_samples(X_test)
y_scores_test

array([0.55604014, 0.492651  , 0.50384849, ..., 0.47248961, 0.49113168,
       0.5364104 ])

In [54]:
# Evaluate
isolation_forest_results = evaluate_model('isolation_forest', y_test, y_pred_test, y_scores_test)

              precision    recall  f1-score   support

           0       0.88      0.88      0.88      7857
           1       0.30      0.30      0.30      1335

    accuracy                           0.79      9192
   macro avg       0.59      0.59      0.59      9192
weighted avg       0.80      0.79      0.80      9192



,AUPRC,AUC,Rec@K
0,0.267307,0.644241,0.301873


In [55]:
# Sklearn classification report
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88      7857
           1       0.30      0.30      0.30      1335

    accuracy                           0.79      9192
   macro avg       0.59      0.59      0.59      9192
weighted avg       0.80      0.79      0.80      9192



## 3. Two step approach
Strategy for graph anomaly detection that first learns graph-aware embeddings and then use traditional anomaly detection techniques from anomaly scoring on enriched vectors.

Graph Convolutional Network (GCN) in the **self-supervised** setting is trained on a **link-prediction** objective - predict whether an edge exists between two nodes.
So link prediction task is here just to learn graph topology aware embeddings.

After training, we extract node embeddings out of GCN and run Isolation Forest on it to flag anomalies — combining graph structure with tabular detection.

**Steps:**
1. Split edges into train/val sets (`RandomLinkSplit`)
2. Define the self-supervised GCN encoder
3. Train with a binary cross-entropy link-prediction loss
4. Extract embeddings and detect anomalies with Isolation Forest

---

##### Building the PyTorch Geometric Graph

To use GNN-based detectors we need a `torch_geometric.Data` object that bundles:
- **`x`** — node feature matrix of shape `(num_nodes, num_features)`
- **`edge_index`** — edge list as a `(2, num_edges)` tensor with contiguous node indices
- **`y`** — ground-truth labels (used only for evaluation, not for training)

In [64]:
# Map edges to contiguous indices using node_mapping and create a (2, num_edges) tensor
edge_index = torch.tensor(
    [[src, dst] for src, dst in edges],
    dtype=torch.long,
).t().contiguous()

# Add undirected edges (since the graph is undirected)
edge_index = utils.to_undirected(edge_index)

# Extract node features — select all columns that start with 'f_'
feature_cols = [c for c in yelp_chi.columns if c.startswith('f_')]
x = torch.tensor(yelp_chi[feature_cols].values, dtype=torch.float)

# Extract labels from the 'spam' column
y = torch.tensor(yelp_chi['spam'].values, dtype=torch.long)

# Create the PyTorch Geometric Data object with x, edge_index, and y
pytorch_data = Data(x=x, edge_index=edge_index, y=y)
pytorch_data

Data(x=[45954, 30], edge_index=[2, 7693958], y=[45954])

In [ ]:
# Apply train/test split to the PyG data object via masks
pytorch_data.train_mask = torch.zeros(pytorch_data.num_nodes, dtype=torch.bool)
pytorch_data.test_mask = torch.zeros(pytorch_data.num_nodes, dtype=torch.bool)
pytorch_data.train_mask[train_idx] = True
pytorch_data.test_mask[test_idx] = True

Data(x=[45954, 30], edge_index=[0], y=[45954], train_mask=[45954], test_mask=[45954])

##### Train/val split

In [ ]:
# Split edges into train/val sets for unsupervised link prediction objective
transform_unsupervised_split = RandomLinkSplit(
    num_val=0.1,
    num_test=0,
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0
)

pytorch_data_unsupervised_train, pytorch_data_unsupervised_val, _ = transform_unsupervised_split(pytorch_data)

pytorch_data_unsupervised_train.to(device)
pytorch_data_unsupervised_val.to(device)

print(pytorch_data_unsupervised_train)
print(pytorch_data_unsupervised_val)

Data(x=[45954, 30], edge_index=[2, 6924564], y=[45954], edge_label=[6924564], edge_label_index=[2, 6924564])
Data(x=[45954, 30], edge_index=[2, 6924564], y=[45954], edge_label=[769394], edge_label_index=[2, 769394])


##### Model

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout=0.3, aggr='mean'):
        super().__init__()
        self.dropout_rate = dropout
        self.convs = torch.nn.ModuleList()
        
        # Stack of Graph Convolutional Network (GCN) layers:
        self.convs.append(SAGEConv(in_channels, hidden_channels, aggr=aggr, project=True))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, aggr=aggr, project=True))
        
        # Linear projection head
        self.head = nn.Linear(hidden_channels, out_channels)
    
    def forward(self, x, edge_index):
        # TASK 2: complete the GCN forward pass
        # - Pass x through each SAGEConv layer in self.convs (don't forget to pass edge_index to each conv)
        # - After each conv, apply activation function and dropout (with parameter training=self.training)
        # - Finally, pass through the linear projection head self.head and return it

        # --- YOUR CODE HERE ---

<details>
<summary><b>Task 2 — Solution example</b></summary>

```python
for conv in self.convs:
    x = conv(x, edge_index)
    x = F.mish(x)
    x = F.dropout(x, p=self.dropout_rate, training=self.training)

return self.head(x)
```

</details>

In [ ]:
class UnsupervisedGCN:
    """Wraps a GNN encoder for unsupervised link-prediction training."""

    def __init__(self, model, lr=0.001):
        self.model = model
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = torch.nn.BCEWithLogitsLoss()

    # Fit the model using the unsupervised link prediction objective
    def fit(self, train_data, val_data=None, epochs=100, log_interval=10):
        device = train_data.x.device
        self.model.to(device)

        # Training loop
        for epoch in range(1, epochs + 1):
            self.model.train()
            self.optimizer.zero_grad()

            z = self.model(train_data.x, train_data.edge_index)
            z = torch.nn.functional.normalize(z, p=2, dim=1)

            src, dst = train_data.edge_label_index
            edge_scores = (z[src] * z[dst]).sum(dim=-1)

            loss = self.criterion(edge_scores, train_data.edge_label)
            loss.backward()
            self.optimizer.step()

            train_acc = self.evaluate(train_data)
            val_acc = self.evaluate(val_data) if val_data else 0.0

            if epoch % log_interval == 0:
                print(f'Epoch: {epoch:03d} | Loss: {loss.item():.4f} | '
                      f'Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

    @torch.no_grad()
    def evaluate(self, data):
        """Evaluate link prediction accuracy on the given data."""
        self.model.eval()
        z = self.model(data.x, data.edge_index)
        z = torch.nn.functional.normalize(z, p=2, dim=1)

        src, dst = data.edge_label_index
        scores = (z[src] * z[dst]).sum(dim=-1)
        
        preds = (torch.sigmoid(scores) > 0.5).float()
        acc = (preds == data.edge_label).float().mean().item()
        return acc

    @torch.no_grad()
    def get_embeddings(self, data):
        """Get node embeddings from the trained model."""
        self.model.eval()
        device = next(self.model.parameters()).device
        x = data.x.to(device)
        edge_index = data.edge_index.to(device)
        z = self.model(x, edge_index)
        z = torch.nn.functional.normalize(z, p=2, dim=1)
        return z.cpu().numpy()

In [58]:
gcn_head = GCN(
    in_channels=pytorch_data.x.shape[1],
    hidden_channels=128,
    out_channels=32,
    num_layers=2,
    dropout=0.2,
    aggr='mean'
)

gcn_model = UnsupervisedGCN(model=gcn_head)

NameError: name 'GCN' is not defined

In [ ]:
gcn_model.fit(pytorch_data_unsupervised_train, pytorch_data_unsupervised_val, epochs=10, log_interval=10)

In [59]:
# After training, we can get the node embeddings from the trained GNN model
gcn_embeddings = gcn_model.get_embeddings(pytorch_data)
print('Embedding Shape:', gcn_embeddings.shape)

NameError: name 'gcn_model' is not defined

In [60]:
# Split embeddings using the same train/test indices
gcn_embeddings_train = gcn_embeddings[train_idx]
gcn_embeddings_test = gcn_embeddings[test_idx]
y_gcn_test = pytorch_data.y.numpy()[test_idx]

NameError: name 'gcn_embeddings' is not defined

In [ ]:
# Anomaly detection with isolation forest on graph-aware embeddings
iso_forest_emb = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
iso_forest_emb.fit(gcn_embeddings_train)

two_step_predictions = (iso_forest_emb.predict(gcn_embeddings_test) == -1).astype(int)
two_step_scores = -iso_forest_emb.score_samples(gcn_embeddings_test)

In [ ]:
two_step_results = evaluate_model(
    'unsupervised_two_step',
    y_gcn_test,
    two_step_predictions,
    two_step_scores,
)

## 4. DOMINANT

[DOMINANT](https://epubs.siam.org/doi/10.1137/1.9781611975673.67) (Deep Anomaly Detection on Attributed Networks) uses a graph autoencoder to jointly reconstruct the adjacency matrix and node features.   Since the vast majority of nodes (reviews) are legitimate, the model learns to encode "normal" patterns of connectivity and features. Fraudulent reviews — with unusual features or atypical neighborhood structure — get reconstructed poorly, so high reconstruction error flags anomalies.


We use the [PyGOD](https://docs.pygod.org/) implementation which handles training and threshold selection.

---

In [ ]:
# TASK 3: defining and fitting DOMINANT model.
# - Start by opening https://docs.pygod.org/ and search for DOMINANT algorithm
# - Define DOMINANT model with main parameters (e.g., hidden dimension, number of layers, epochs, etc.)
# - Do not forget assigning contamination parameter and verbose=3 level to the model
# - Fit the model on the PyTorch Geometric data object (pytorch_data)

dominant = # --- DEFINE DOMINANT MODEL HERE ---


dominant = DOMINANT(
    contamination=contamination,
    hid_dim=16,
    num_layers=2,
    epoch=100,
    verbose=3,
    gpu=0
)

dominant.fit(pytorch_data)


<details>
<summary><b>Task 3 — Solution example</b></summary>

```python
dominant = DOMINANT(
    contamination=contamination,
    hid_dim=16,
    num_layers=2,
    epoch=100,
    verbose=3,
)

dominant.fit(pytorch_data)
```

</details>

In [ ]:
# Get anomaly scores and binary predictions from the trained DOMINANT model
dominant_scores = dominant.decision_function(pytorch_data)
dominant_labels = dominant.predict(pytorch_data)

In [ ]:
dominant_results = evaluate_model(
    'dominant',
    pytorch_data.y[pytorch_data.test_mask].numpy(),
    dominant_labels[pytorch_data.test_mask],
    dominant_scores[pytorch_data.test_mask],
)

## 5. GAD-NR

[GAD-NR](https://arxiv.org/abs/2306.01951) (Graph Anomaly Detection via Neighbourhood Reconstruction) learns to reconstruct the neighbourhood of each node. Nodes whose neighbourhoods are hard to reconstruct are marked as anomalous.

We wrap the PyGOD `GADNR` detector with a custom backbone (GCN) to control the architecture.

---

In [ ]:
def gadnr_model(
    batch_size, contamination, verbose, hid_dim, epoch, num_layers
):
    def gcn(**kwargs):
        kwargs.pop('tot_nodes')
        return nn.GCN(**kwargs)

    if batch_size == 0:
        batch_size = 64

    return GADNR(
        backbone=gcn,
        hid_dim=hid_dim,
        batch_size=batch_size,
        epoch=epoch,
        contamination=contamination,
        verbose=verbose,
        num_layers=num_layers,
        dropout=0.1,
        gpu=0
    )

In [ ]:
gadnr = gadnr_model(
    hid_dim=16, epoch=50, verbose=3, batch_size=4096, contamination=contamination, num_layers=2
)

gadnr.fit(pytorch_data)

In [ ]:
# Get anomaly scores and binary predictions from the trained GADNR model
gadnr_scores = gadnr.decision_function(pytorch_data)
gadnr_labels = gadnr.predict(pytorch_data)

In [ ]:
# Evaluate
gadnr_results = evaluate_model(
    'gadnr',
    pytorch_data.y[pytorch_data.test_mask].numpy(),
    gadnr_labels[pytorch_data.test_mask],
    gadnr_scores[pytorch_data.test_mask],
)

## 6. Model Comparison

Let's compare all four models side-by-side. Which approach benefits most from graph structure?

---

In [62]:
models_results = compare_models([
    isolation_forest_results,
    two_step_results,
    dominant_results,
    gadnr_results,
])

NameError: name 'two_step_results' is not defined